In [1]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup

def scrape_business_info(url):
    # Send a request to the website
    response = requests.get(url)
    response.raise_for_status()  # Check if the request was successful

    # Parse the HTML content
    soup = BeautifulSoup(response.content, 'html.parser')

    # Example selectors - adjust these according to the structure of the target website
    business_name_selector = 'strong'  # Replace with the actual CSS selector
    address_selector = 'div.address'             # Replace with the actual CSS selector
    phone_number_selector = 'div.phone-number'   # Replace with the actual CSS selector

    # Extract the information
    business_name = soup.select_one(business_name_selector).get_text(strip=True)
    address = soup.select_one(address_selector).get_text(strip=True)
    phone_number = soup.select_one(phone_number_selector).get_text(strip=True)

    return {
        'Business Name': business_name,
        'Address': address,
        'Phone Number': phone_number
    }

# URL of the website to scrape
url = 'https://ccdenver.org/marisol-family/'  # Replace with the actual URL

# Scrape the information
business_info = scrape_business_info(url)

# Print the scraped information
print("Business Name:", business_info['Business Name'])
print("Address:", business_info['Address'])
print("Phone Number:", business_info['Phone Number'])


AttributeError: 'NoneType' object has no attribute 'get_text'

In [ ]:
# Import necessary libraries
import requests
from bs4 import BeautifulSoup
import re

In [ ]:
# Function to extract email addresses from a given URL
def extract_emails(url):
    try:
        # Send a request to the website
        response = requests.get(url)

        # Check if the request was successful
        if response.status_code == 200:
            # Parse the HTML content of the page
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract text from the parsed HTML
            text = soup.get_text()

            # Use regular expression to find email addresses
            emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)

            # Remove duplicate emails
            emails = list(set(emails))

            return emails
        else:
            return f"Failed to retrieve the webpage. Status code: {response.status_code}"
    except Exception as e:
        return str(e)

In [ ]:
# URL of the website you want to scrape

# Abortion
# https://cobaltaf.org Uses an email form
# https://www.coloradodoulaproject.org/contact-us returns 3 emails
# https://cwhccolorado.com/appointments/index.html only has phone number
# https://healthyfuturesabortion.com/ -- A status code of 403 means that access to the requested resource is forbidden

# Food
# https://anchorofhopedenver.wixsite.com/ministry -- Would need to crawl to contact us page
# https://www.wellpower.org/contact/ -- launches email app to send email to Press Contact (media inquireies)
# https://www.arapahoeco.gov/your_county/county_departments/human_services/index.php -- email exists on contact us page

# Next step: add in navigation to 'contact us' page

url = 'https://www.arapahoeco.gov/your_county/county_departments/human_services/index.php'  # Replace with the actual URL

# Extract and print emails
emails = extract_emails(url)
print(f"Emails found: {emails}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

In [ ]:
def extract_phone_numbers(text):
    return re.findall(r'\+?\d[\d -]{8,12}\d', text)

def get_internal_links(base_url, soup):
    internal_links = set()
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        full_url = urljoin(base_url, href)
        if base_url in full_url:
            internal_links.add(full_url)
    return internal_links

def fetch_page(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return BeautifulSoup(response.content, 'html.parser')
    return None

def crawl_website(base_url):
    visited = set()
    to_visit = [base_url]
    phone_numbers = set()
    contact_page = None

    while to_visit:
        url = to_visit.pop(0)
        if url in visited:
            continue
        visited.add(url)
        soup = fetch_page(url)
        if not soup:
            continue

        text = soup.get_text()
        found_phone_numbers = extract_phone_numbers(text)
        if found_phone_numbers:
            print(f"Found phone numbers on {url}: {found_phone_numbers}")
            phone_numbers.update(found_phone_numbers)

        if 'contact' in url.lower() or 'contact' in text.lower() or 'contact info' in text.lower():
            contact_page = url

        internal_links = get_internal_links(base_url, soup)
        to_visit.extend(internal_links - visited)

    # Prioritize scraping the contact page
    if contact_page:
        print(f"Checking contact page: {contact_page}")
        soup = fetch_page(contact_page)
        if soup:
            text = soup.get_text()
            found_phone_numbers = extract_phone_numbers(text)
            if found_phone_numbers:
                print(f"Found phone numbers on contact page {contact_page}: {found_phone_numbers}")
                phone_numbers.update(found_phone_numbers)

    return phone_numbers

In [ ]:
# URL of the website to scrape
base_url = 'https://anchorofhopedenver.wixsite.com/ministry'

# Crawl the website and extract phone numbers
phone_numbers = crawl_website(base_url)
print(f"Phone numbers found: {phone_numbers}")